## Detección de Anomalías No Supervisada

**Objetivo:** Implementar modelos no supervisados sin hacer referencia a la variable objetivo (is_fraud) para detectar patrones de anomalías nuevos y no etiquetados (Fraude Zero-Day).

<details>
<summary><b>Justificación Metedológica</b></summary>

Depender exclusivamente de modelos supervisados limita la detección a tipos de fraude ya conocidos. Esta etapa procesa los datos en bruto directamente (fraudTrain.csv y fraudTest.csv) en lugar de reutilizar las variables del enfoque supervisado por las siguientes razones:

1. **Incompatibilidad de Objetivos:** El preprocesamiento supervisado optimiza las variables para divisiones basadas en árboles contra etiquetas. Los modelos no supervisados (Isolation Forest, LOF) dependen de la geometría espacial, distancias y densidad de puntos.
2. **Variables Centradas en Anomalías:** Los algoritmos no supervisados funcionan mejor con variables que enfatizan comportamientos extremos (Codificación por Frecuencia, dinámica temporal cíclica, desviaciones métricas).
3. **Estructura Espacial:** Preserva la firma original de la anomalía sin suavizar las señales de valores atípicos (outliers).
</details>

In [ ]:
# Importar las librerías necesarias y funciones de utilidad
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

from src.utils import preprocess_for_anomaly_detection

In [2]:
# Cargar los conjuntos de datos
df_train = pd.read_csv('../data/fraudTrain.csv')
df_test = pd.read_csv('../data/fraudTest.csv')

# Separar variable objetivo para los modelos no supervisados
X_train_raw = df_train.drop(columns=['is_fraud'])
y_train_true = df_train['is_fraud']

X_test_raw = df_test.drop(columns=['is_fraud'])
y_test_true = df_test['is_fraud']

print("X_train_raw dimensions:", X_train_raw.shape)
print("X_test_raw dimensions:", X_test_raw.shape)

X_train_raw dimensions: (1296675, 22)
X_test_raw dimensions: (555719, 22)


### Criterios de Eliminación de Variables

Se descartaron las siguientes variables, agrupadas por motivo:

* **Identificadores únicos y PII:** (`Unnamed: 0`, `cc_num`, `first`, `last`, `street`, `trans_num`): no representan patrones de fraude generalizables — fuerzan al modelo a memorizar individuos en vez de comportamientos.
* **Texto de alta cardinalidad:** (`job`, `city`, `state`, `merchant`, `category`): reemplazadas por Frequency Encoding evitando la explosión dimensional de un One-Hot Encoding con miles de categorías únicas.
* **Coordenadas crudas** (`lat`, `long`, `merch_lat`, `merch_long`): reemplazadas por la distancia Haversine (`distance_km`) entre cliente y comercio, una variable más compacta y con mayor señal directa de comportamiento anómalo.
* **Timestamp crudo y fecha de nacimiento:** (`trans_date_trans_time`, `dob`): ya extraída su información útil en variables derivadas (`hour_sin`, `hour_cos`, `age`).
* **Variables adicionales sin valor predictivo claro:** (`zip`, `unix_time`, `city_pop`): descartadas por redundancia o poca importancia.

In [3]:
# Preprocesar train (genera los mapas de frecuencia)
X_train, freq_maps = preprocess_for_anomaly_detection(X_train_raw)

# Preprocesar test usando los MISMOS mapas de frecuencia de train (sin data leakage)
X_test, _ = preprocess_for_anomaly_detection(X_test_raw, freq_maps=freq_maps)

print("X_train columns:", X_train.columns.tolist())
print("X_test columns:", X_test.columns.tolist())

X_train columns: ['amt', 'gender', 'city_pop', 'hour_sin', 'hour_cos', 'distance_km', 'age', 'category_freq', 'job_freq', 'state_freq', 'merchant_freq']
X_test columns: ['amt', 'gender', 'city_pop', 'hour_sin', 'hour_cos', 'distance_km', 'age', 'category_freq', 'job_freq', 'state_freq', 'merchant_freq']


### Justificación del Algoritmo

Se seleccionó **Isolation Forest** como primer enfoque no supervisado por las siguientes razones:

* **Desbalance Extremo:** El fraude representa solo ~0.52% del volumen total, por lo que un modelo de aislamiento resulta más adecuado que otros métodos de detección de outliers.
* **Aislamiento Geométrico:** A diferencia de métodos basados en densidad, Isolation Forest aísla anomalías mediante particiones aleatorias del espacio de características, sin necesitar etiquetas previas — coherente con el enfoque no supervisado del experimento.
* **Escalabilidad:** Su complejidad computacional lo hace eficiente para procesar datasets grandes (~1.8M de registros combinados entre train y test).

In [4]:
# Experimento 1: Entrenar solo con transacciones legítimas

mask_legit_train = (y_train_true == 0)
X_legit_train = X_train[mask_legit_train]

iso_forest = IsolationForest(
    n_estimators=300, contamination=0.01, random_state=42, n_jobs=-1
)
iso_forest.fit(X_legit_train)

# Evaluar sobre TEST completo (legítimas + fraude)
preds_test = iso_forest.predict(X_test)
preds_test_binary = [1 if p == -1 else 0 for p in preds_test]

# Hacer una tupla con el valor predicho  y con el valor real
captured_frauds = sum(
    1 for pred, real in zip(preds_test_binary, y_test_true) if pred == 1 and real == 1
)
total_frauds_test = (y_test_true == 1).sum()

print(f"Total fraud cases (test): {total_frauds_test:,}")
print(f"Frauds detected: {captured_frauds:,}")
print(f"Fraud detection rate (Recall): {(captured_frauds / total_frauds_test) * 100:.2f}%")

Total fraud cases (test): 2,145
Frauds detected: 42
Fraud detection rate (Recall): 1.96%


In [5]:
# Experimento 2: Entrenar con TODO train (legítimas + fraude), evaluar en TEST

iso_forest_total = IsolationForest(
    n_estimators=300, contamination=0.01, random_state=42, n_jobs=-1
)
iso_forest_total.fit(X_train)  # entrena solo con train

preds_test_total = iso_forest_total.predict(X_test)  # evalúa solo con test
preds_test_total_binary = [1 if p == -1 else 0 for p in preds_test_total]

captured_frauds_total = sum(
    1 for pred, real in zip(preds_test_total_binary, y_test_true) if pred == 1 and real == 1
)

print(f"Total fraud cases (test): {total_frauds_test:,}")
print(f"Frauds detected by the model: {captured_frauds_total:,}")
print(f"Fraud detection rate (Recall): {(captured_frauds_total / total_frauds_test) * 100:.2f}%")

Total fraud cases (test): 2,145
Frauds detected by the model: 45
Fraud detection rate (Recall): 2.10%


**Resultados Finales:**

| Enfoque | Fraudes Detectados | Recall |
|---|---|---|
| Entrenado solo con transacciones legítimas | 42 / 2,145 | 1.96% |
| Entrenado con dataset completo (legítimas + fraude) | 45 / 2,145 | 2.10% |

**Conclusión:** Entrenar Isolation Forest exclusivamente con transacciones legítimas produjo un recall extremadamente bajo (1.96%), y entrenarlo con el dataset completo apenas mejoró el resultado (2.10%) — confirmando que el modelo no logra distinguir eficazmente el fraude real de la variabilidad normal de los datos legítimos. Este resultado,confirma que Isolation Forest no resulta adecuado para la topología de este espacio de características, debido al alto solapamiento entre transacciones normales y fraudulentas en el espacio geométrico que el algoritmo utiliza para aislar anomalías. En otras palabras, no logra detectar patrones claro de fraude o transacciones legítimas para aislarlas correctamente.